In [215]:
import sys

sys.path.append("..")

import polars as pl
import pronouncing

from phoneme import split_words
from tqdm import tqdm


class Counter:
    def __init__(self):
        self.map = {}

    def add(self, key):
        if key in self.map:
            self.map[key] += 1
        else:
            self.map[key] = 1

    def union(self, counter):
        for k, v in counter.map.items():
            if k in self.map:
                self.map[k] += v
            else:
                self.map[k] = v

    def find(self, key, cutoff=3):
        if key in self.map and self.map[key] > cutoff:
            return self.map[key]

        return None


def get_prefixs(words: list[str]) -> Counter:
    prefixes = Counter()
    for word in words:
        variations = pronouncing.phones_for_word(word)
        for variant in variations:
            prefix = get_prefix(variant)
            if prefix != "":
                prefixes.add(prefix)

    return prefixes


def get_prefix(arpabet_word: str) -> str:
    consonants = []
    for phoneme in arpabet_word.split():
        if phoneme.strip("012") != phoneme:
            break

        consonants.append(phoneme)

    return " ".join(consonants)


df = pl.read_ndjson("../data/wikipedia_articles.jsonl")
prefixes = Counter()
for row in tqdm(df.iter_rows(named=True), desc="computing prefixes", total=df.shape[0]):
    content = row["content"]
    words = split_words(content)
    assert len(words) > 0

    new_prefixes = get_prefixs(words)
    prefixes.union(new_prefixes)

computing prefixes: 100%|██████████| 1000/1000 [00:11<00:00, 90.71it/s]


In [216]:
total_prefixes = sum([t for _, t in prefixes.map.items()])
print(f"unique prefixes: {len(prefixes.map)} total prefixes: {total_prefixes:,}")

for i in range(1, 10, 1):
    normal_prefixes = {k: v for k, v in prefixes.map.items() if v > i}
    print(f"unique prefixes with more than {i} appearance(s): {len(normal_prefixes)}")

unique prefixes: 104 total prefixes: 4,617,234
unique prefixes with more than 1 appearance(s): 99
unique prefixes with more than 2 appearance(s): 96
unique prefixes with more than 3 appearance(s): 94
unique prefixes with more than 4 appearance(s): 94
unique prefixes with more than 5 appearance(s): 90
unique prefixes with more than 6 appearance(s): 87
unique prefixes with more than 7 appearance(s): 87
unique prefixes with more than 8 appearance(s): 86
unique prefixes with more than 9 appearance(s): 83


In [ ]:
sonority_map = {
    # Vowels (highest sonority)
    "AA": 6,
    "AE": 6,
    "AH": 6,
    "AO": 6,
    "AW": 6,
    "AY": 6,
    "EH": 6,
    "ER": 6,
    "EY": 6,
    "IH": 6,
    "IY": 6,
    "OW": 6,
    "OY": 6,
    "UH": 6,
    "UW": 6,
    # Glides (semi-vowels)
    "W": 5,
    "Y": 5,
    # Liquids
    "L": 4,
    "R": 4,
    # Nasals
    "M": 3,
    "N": 3,
    "NG": 3,
    # Fricatives
    "DH": 2,
    "F": 2,
    "V": 2,
    "TH": 2,
    "S": 2,
    "Z": 2,
    "SH": 2,
    "ZH": 2,
    "HH": 2,
    # Plosives (Stops)
    "B": 1,
    "D": 1,
    "G": 1,
    "K": 1,
    "P": 1,
    "T": 1,
    "CH": 1,
    "JH": 1,
    "START": -1,
    "END": -1,
}


def get_sonority(phoneme: str) -> int:
    return sonority_map[phoneme]


def is_valley(left: int, center: int, right: int) -> bool:
    return left > center and center < right


def is_peek(left: int, center: int, right: int) -> bool:
    return left < center and center > right


def is_vowel(phoneme: str) -> bool:
    return phoneme.strip("012") != phoneme


def iter_word(word: list[str]):
    for i in range(len(word)):

        left = "START"
        if i - 1 >= 0:
            left = word[i - 1]

        right = "END"
        if i + 1 <= len(word) - 1:
            right = word[i + 1]

        yield (left, word[i], right)


class Word:
    def __init__(self):
        self.syllables = []

    def insert(self, syllable):
        if syllable.nucleus != "":
            self.syllables.insert(0, syllable)

        if syllable.nucleus == "":
            [self.syllables[0].insert(p) for p in syllable.coda]

    def __repr__(self):
        return "<" + " ".join([repr(s) for s in self.syllables]) + ">"


class Syllable:
    def __init__(self, phoneme=None):
        self.onset = []
        self.nucleus = ""
        self.coda = []

        if phoneme is None:
            return

        if is_vowel(phoneme):
            self.nucleus = phoneme
        else:
            self.coda.append(phoneme)

    def has_nucleus(self):
        return self.nucleus != ""

    def insert(self, phoneme):
        if is_vowel(phoneme):
            self.nucleus = phoneme
        elif self.nucleus == "":
            self.coda.insert(0, phoneme)
        else:
            self.onset.insert(0, phoneme)

    def __repr__(self):
        onset = " ".join(self.onset)
        if onset != "":
            onset += "."

        coda = " ".join(self.coda)
        if coda != "":
            coda = "." + coda

        return f"[{onset}{self.nucleus}{coda}]"


def get_syllables(word: list[str]) -> list[list[str]]:
    assert isinstance(word, list)

    final_word = Word()
    syllable = Syllable()
    iter = iter_word(word)
    for left, center, right in reversed(list(iter)):
        is_cons = not is_vowel(center)

        # this is a vowel and the syllable does not yet have a nucleus
        if not is_cons and not syllable.has_nucleus():
            syllable.insert(center)

            continue

        # this is a vowel and the current syllable does have a nucleus
        if not is_cons and syllable.has_nucleus():
            final_word.insert(syllable)
            syllable = Syllable(center)

            continue

        # this is not a vowel and the syllable does not yet have a nucleus
        if is_cons and not syllable.has_nucleus():
            syllable.insert(center)

            continue

        # check if this is not a valid english prefix meaning we should break the syllable here
        prefix = " ".join([center] + syllable.onset)
        if not prefixes.find(prefix):
            final_word.insert(syllable)
            syllable = Syllable(center)

            continue

        # center is a consonant and the syllable has a nucleus
        if is_valley(left, center, right):
            syllable.insert(center)
            final_word.insert(syllable)
            syllable = Syllable()

            continue

        # handle the weird edge case where 'S' breaks the SSP in english
        if is_peek(left, center, right) and center == "S":
            syllable.insert(center)
            final_word.insert(syllable)
            syllable = Syllable()

            continue

        syllable.insert(center)

    final_word.insert(syllable)

    return final_word


tests = [
    ("championship", "<[CH.AE1.M] [P.IY0] [AH0.N] [SH.IH2.P]>"),
    ("tournament's", "<[T.UH1.R] [N.AH0] [M.AH0.N T S]>"),
    ("can't", "<[K.AE1.N T]>"),
    ("won't", "<[W.OW1.N T]>"),
    ("shouldn't", "<[SH.UH1] [D.AH0.N T]>"),
    ("McDonald", "<[M.AH0.K] [D.AA1] [N.AH0.L D]>"),
    ("D'Angelo", "<[D.IY0] [AE1.N] [JH.IH0] [L.OW0]>"),
    ("iPhone", "<[AY1] [F.OW2.N]>"),
    ("eBay", "<[IY1] [B.EY2]>"),
    ("CoPilot", "<[K.OW1] [P.AY0] [L.AH0.T]>"),
    ("fire", "<[F.AY1] [ER0]>"),
    ("hour", "<[AW1] [ER0]>"),
    ("player", "<[P L.EY1] [ER0]>"),
    ("knight", "<[N.AY1.T]>"),
    ("aisle", "<[AY1.L]>"),
    ("separate", "<[S.EH1] [P.ER0] [EY2.T]>"),
    ("camera", "<[K.AE1] [M.ER0] [AH0]>"),
    ("family", "<[F.AE1] [M.AH0] [L.IY0]>"),
    ("strength", "<[S T R.EH1.NG K TH]>"),
    ("through", "<[TH R.UW1]>"),
    ("blasts", "<[B L.AE1.S T S]>"),
    ("wallpaper", "<[W.AO1.L] [P.EY2] [P.ER0]>"),
    ("football", "<[F.UH1.T] [B.AO2.L]>"),
    ("bookshelf", "<[B.UH1.K] [SH.EH2.L F]>"),
    ("guitar", "<[G.IH0] [T.AA1.R]>"),
    ("police", "<[P.AH0] [L.IY1.S]>"),
    ("croissant", "<[K W.AA2] [S.AA1.N T]>"),
    ("cliche", "<[K L.IY0] [SH.EY1]>"),
    ("subtle", "<[S.AH1] [T.AH0.L]>"),
    ("psalm", "<[S.AA1.L M]>"),
    ("sphere", "<[S F.IH1.R]>"),
    ("split", "<[S P L.IH1.T]>"),
    ("sphinx", "<[S F.IH1.NG K S]>"),
    ("sprint", "<[S P R.IH1.N T]>"),
    ("sprints"),
    ("thrive"),
    ("free"),
    ("twelfth"),
    ("lengths"),
    # ("bookworm", "<[B.UH1.K] [w.ER2.M]>"), # taking this out as this will currently fail
]
for test in tests:
    variations = pronouncing.phones_for_word(test[0])
    variation = variations[0]
    word = get_syllables(variation.split())

    print(f"{test[0]}\t{variation}\t{word}")
    assert repr(word) == test[1], f"\n'{word}' != \n'{test[1]}'"

championship	CH AE1 M P IY0 AH0 N SH IH2 P	<[CH.AE1.M] [P.IY0] [AH0.N] [SH.IH2.P]>
tournament's	T UH1 R N AH0 M AH0 N T S	<[T.UH1.R] [N.AH0] [M.AH0.N T S]>
can't	K AE1 N T	<[K.AE1.N T]>
won't	W OW1 N T	<[W.OW1.N T]>
shouldn't	SH UH1 D AH0 N T	<[SH.UH1] [D.AH0.N T]>
McDonald	M AH0 K D AA1 N AH0 L D	<[M.AH0.K] [D.AA1] [N.AH0.L D]>
D'Angelo	D IY0 AE1 N JH IH0 L OW0	<[D.IY0] [AE1.N] [JH.IH0] [L.OW0]>
iPhone	AY1 F OW2 N	<[AY1] [F.OW2.N]>
eBay	IY1 B EY2	<[IY1] [B.EY2]>
CoPilot	K OW1 P AY0 L AH0 T	<[K.OW1] [P.AY0] [L.AH0.T]>
fire	F AY1 ER0	<[F.AY1] [ER0]>
hour	AW1 ER0	<[AW1] [ER0]>
player	P L EY1 ER0	<[P L.EY1] [ER0]>
knight	N AY1 T	<[N.AY1.T]>
aisle	AY1 L	<[AY1.L]>
separate	S EH1 P ER0 EY2 T	<[S.EH1] [P.ER0] [EY2.T]>
camera	K AE1 M ER0 AH0	<[K.AE1] [M.ER0] [AH0]>
family	F AE1 M AH0 L IY0	<[F.AE1] [M.AH0] [L.IY0]>
strength	S T R EH1 NG K TH	<[S T R.EH1.NG K TH]>
through	TH R UW1	<[TH R.UW1]>
blasts	B L AE1 S T S	<[B L.AE1.S T S]>
wallpaper	W AO1 L P EY2 P ER0	<[W.AO1.L] [P.EY2] [P.ER0]>
footb

In [ ]:
df = pl.read_ndjson("../data/wikipedia_articles.jsonl")
onsets = Counter()
codas = Counter()
nucleus = Counter()

def has_vowel(word: str):
    return "0" in word or "1" in word or "2" in word

for row in tqdm(df.iter_rows(named=True), desc="computing onsets, nucleus, and codas", total=df.shape[0]):
    content = row["content"]
    words = split_words(content)
    assert len(words) > 0

    for word in words:
        variations = pronouncing.phones_for_word(word)
        for variation in variations:
            # skip words without a vowel as they don't really work with the syllable detection
            if not has_vowel(variation):
                continue

            # some variations have annotations on them like `# abbrev` just skip those for now
            if "#" in variation:
                continue

            final_word = get_syllables(variation.split())
                
            for syllable in final_word.syllables:
                onset = " ".join(syllable.onset)
                if onset != "":
                    onsets.add(onset)

                nucleus.add(syllable.nucleus[:2])

                coda = " ".join(syllable.coda)
                assert coda.upper() == coda, f"word is '{word}', vairation is '{variation}' coda is '{coda}'"
                if coda != "":
                    codas.add(coda)
                

computing onsets, nucleus, and codas: 100%|██████████| 1000/1000 [00:43<00:00, 22.82it/s]


In [230]:
total_nucleus = sum([v for _, v in nucleus.map.items()])
print(f"unique nucleus: {len(nucleus.map)}, total nucleus: {total_nucleus:,}")
print([k for k in nucleus.map.keys()])

print("---")
total_onsets = sum([v for _, v in onsets.map.items()])
print(f"unique onsets: {len(onsets.map)}, total onsets: {total_onsets:,}")
for i in range(1, 5, 1):
    filtered_onsets = [k for k in onsets.map.keys() if len(k.split()) == i]
    print(f"unique onsets with {i} consonants {len(filtered_onsets)}")

print("---")
total_codas = sum([v for _, v in codas.map.items()])
print(f"unique codas: {len(codas.map)}, total codas: {total_codas:,}")
for i in range(1, 6, 1):
    filtered_codas = [k for k in codas.map.keys() if len(k.split()) == i]
    print(f"unique codas with {i} consonants {len(filtered_codas)}")

unique nucleus: 15, total nucleus: 9,542,989
['AH', 'AE', 'IH', 'AO', 'ER', 'IY', 'UW', 'OW', 'AY', 'EH', 'AA', 'UH', 'EY', 'AW', 'OY']
---
unique onsets: 103, total onsets: 7,503,374
unique onsets with 1 consonants 23
unique onsets with 2 consonants 70
unique onsets with 3 consonants 10
unique onsets with 4 consonants 0
---
unique codas: 228, total codas: 4,745,233
unique codas with 1 consonants 22
unique codas with 2 consonants 106
unique codas with 3 consonants 92
unique codas with 4 consonants 8
unique codas with 5 consonants 0


In [231]:
data = []
for onset, count in onsets.map.items(): 
    data.append({
        "onset": onset ,
        "count": count,
    })

pl.DataFrame(data).sort(pl.col("count")).write_ndjson("../data/onsets.jsonl")

data = []
for n, count in nucleus.map.items(): 
    data.append({
        "nucleus": n,
        "count": count,
    })

pl.DataFrame(data).sort(pl.col("count")).write_ndjson("../data/nucleus.jsonl")

data = []
for coda, count in codas.map.items(): 
    data.append({
        "coda": coda,
        "count": count,
    })

pl.DataFrame(data).sort(pl.col("count")).write_ndjson("../data/coda.jsonl")